# Movie Expert Agent

OpenAI Function Calling을 활용한 영화 정보 에이전트

In [13]:
import os
import json
import requests
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

BASE_URL = 'https://nomad-movies.nomadcoders.workers.dev'

print('Setup complete!')

Setup complete!


In [14]:
def get_popular_movies():
    response = requests.get(BASE_URL + '/movies')
    return response.json()


def get_movie_details(id):
    response = requests.get(BASE_URL + '/movies/' + str(id))
    return response.json()


def get_movie_credits(id):
    response = requests.get(BASE_URL + '/movies/' + str(id) + '/credits')
    return response.json()


print('Functions defined!')

Functions defined!


In [15]:
tools = [
    {
        'type': 'function',
        'function': {
            'name': 'get_popular_movies',
            'description': '현재 인기 있는 영화 목록을 가져옵니다. 인기 영화나 최신 영화가 무엇인지 물어볼 때 사용하세요.',
            'parameters': {
                'type': 'object',
                'properties': {},
                'required': []
            }
        }
    },
    {
        'type': 'function',
        'function': {
            'name': 'get_movie_details',
            'description': '특정 영화 ID로 영화의 상세 정보(제목, 줄거리, 평점, 장르 등)를 가져옵니다.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'id': {
                        'type': 'integer',
                        'description': '조회할 영화의 ID'
                    }
                },
                'required': ['id']
            }
        }
    },
    {
        'type': 'function',
        'function': {
            'name': 'get_movie_credits',
            'description': '특정 영화 ID로 영화의 출연진(배우)과 제작진(감독, 작가 등) 정보를 가져옵니다.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'id': {
                        'type': 'integer',
                        'description': '조회할 영화의 ID'
                    }
                },
                'required': ['id']
            }
        }
    }
]

print('Tools defined!')

Tools defined!


In [16]:
def run_agent(user_message):
    sep = '=' * 60
    print(sep)
    print('사용자: ' + user_message)
    print(sep)

    messages = [
        {
            'role': 'system',
            'content': (
                '당신은 영화 전문가 에이전트입니다. '
                '사용자의 영화 관련 질문에 답하기 위해 제공된 함수들을 활용하세요. '
                'get_popular_movies로 인기 영화 목록을, '
                'get_movie_details로 특정 영화의 상세 정보를, '
                'get_movie_credits로 특정 영화의 출연진 및 제작진 정보를 조회할 수 있습니다. '
                '답변은 한국어로 친절하게 제공하세요.'
            )
        },
        {'role': 'user', 'content': user_message}
    ]

    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=messages,
        tools=tools,
        tool_choice='auto'
    )

    message = response.choices[0].message

    while message.tool_calls:
        messages.append(message)

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)

            print('[함수 호출] ' + function_name + ' ' + str(arguments))

            if function_name == 'get_popular_movies':
                result = get_popular_movies()
            elif function_name == 'get_movie_details':
                result = get_movie_details(**arguments)
            elif function_name == 'get_movie_credits':
                result = get_movie_credits(**arguments)
            else:
                result = {'error': 'Unknown function: ' + function_name}

            messages.append({
                'role': 'tool',
                'tool_call_id': tool_call.id,
                'content': json.dumps(result, ensure_ascii=False)
            })

        response = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=messages,
            tools=tools,
            tool_choice='auto'
        )
        message = response.choices[0].message

    print()
    print('에이전트: ' + message.content)
    return message.content

In [17]:
# 테스트 1: 인기 영화 조회
run_agent('지금 인기 있는 영화가 무엇인지 알려줘')

사용자: 지금 인기 있는 영화가 무엇인지 알려줘
[함수 호출] get_popular_movies {}

에이전트: 현재 인기 있는 영화 목록은 다음과 같습니다:

1. **당신의 마음이 부서질 겁니다** (Your Heart Will Be Broken)
   - 개요: 고등학생 폴리나는 새로운 학교에서 괴롭힘을 당하다가 주요 괴롭힘꾼 바르스와 거래를 하게 됩니다. 그는 폴리나의 남자친구로 가장하고 그녀를 보호해야 하고, 폴리나는 그의 모든 요구를 들어줘야 합니다. 이 게임 중에 그들은 진정한 감정을 발전시키지만, 가족과 동급생들이 이들의 사랑을 방해하게 됩니다.
   - 평점: 6.6
   - 개봉일: 2026-03-26
   - ![포스터](https://image.tmdb.org/t/p/w780/iGpMm603GUKH2SiXB2S5m4sZ17t.jpg)

2. **아바타: 불과 재** (Avatar: Fire and Ash)
   - 개요: Jake Sully와 Neytiri는 새로운 위협인 Ash People과 맞서 싸우며, 그들의 생존과 Pandora의 미래를 위한 갈등에서 감정적 및 육체적 한계에 도전해야 합니다.
   - 평점: 7.4
   - 개봉일: 2025-12-17
   - ![포스터](https://image.tmdb.org/t/p/w780/bRBeSHfGHwkEpImlhxPmOcUsaeg.jpg)

3. **호퍼스** (Hoppers)
   - 개요: 과학자들이 인간의 의식을 사실적인 로봇 동물로 전이하는 방법을 발견하고, 동물 애호가인 메이블은 이 기술을 사용하여 동물의 세계에서 미스터리를 발견하게 됩니다.
   - 평점: 7.6
   - 개봉일: 2026-03-04
   - ![포스터](https://image.tmdb.org/t/p/w780/xjtWQ2CL1mpmMNwuU5HeS4Iuwuu.jpg)

4. **범죄 101** (Crime 101)
   - 개요: LA의 상징적인 101 고속도로에서 펼쳐지는 높은 위험의 강도 사건에서,

'현재 인기 있는 영화 목록은 다음과 같습니다:\n\n1. **당신의 마음이 부서질 겁니다** (Your Heart Will Be Broken)\n   - 개요: 고등학생 폴리나는 새로운 학교에서 괴롭힘을 당하다가 주요 괴롭힘꾼 바르스와 거래를 하게 됩니다. 그는 폴리나의 남자친구로 가장하고 그녀를 보호해야 하고, 폴리나는 그의 모든 요구를 들어줘야 합니다. 이 게임 중에 그들은 진정한 감정을 발전시키지만, 가족과 동급생들이 이들의 사랑을 방해하게 됩니다.\n   - 평점: 6.6\n   - 개봉일: 2026-03-26\n   - ![포스터](https://image.tmdb.org/t/p/w780/iGpMm603GUKH2SiXB2S5m4sZ17t.jpg)\n\n2. **아바타: 불과 재** (Avatar: Fire and Ash)\n   - 개요: Jake Sully와 Neytiri는 새로운 위협인 Ash People과 맞서 싸우며, 그들의 생존과 Pandora의 미래를 위한 갈등에서 감정적 및 육체적 한계에 도전해야 합니다.\n   - 평점: 7.4\n   - 개봉일: 2025-12-17\n   - ![포스터](https://image.tmdb.org/t/p/w780/bRBeSHfGHwkEpImlhxPmOcUsaeg.jpg)\n\n3. **호퍼스** (Hoppers)\n   - 개요: 과학자들이 인간의 의식을 사실적인 로봇 동물로 전이하는 방법을 발견하고, 동물 애호가인 메이블은 이 기술을 사용하여 동물의 세계에서 미스터리를 발견하게 됩니다.\n   - 평점: 7.6\n   - 개봉일: 2026-03-04\n   - ![포스터](https://image.tmdb.org/t/p/w780/xjtWQ2CL1mpmMNwuU5HeS4Iuwuu.jpg)\n\n4. **범죄 101** (Crime 101)\n   - 개요: LA의 상징적인 101 고속도로에서 펼쳐지는 높은 위험의 강도 사건에서, 탐정이 이 사건을 해결하기 위해 접근하면서 상황은 더욱 위험해집니다.\n 

In [18]:
# 테스트 2: 특정 영화 상세 정보 조회
run_agent('movie ID 550에 해당하는 영화가 무엇인지 알려줘')

사용자: movie ID 550에 해당하는 영화가 무엇인지 알려줘
[함수 호출] get_movie_details {'id': 550}

에이전트: 영화 ID 550에 해당하는 영화는 **"Fight Club"**입니다.  

- **개요**: 한 시간에 대해 고민하는 불면증 환자와 미끄러지듯한 비누 판매자가 남성의 본능적인 공격성을 새로운 형태의 치료법으로 channeling합니다. 그들의 개념은 각 도시마다 underground "격투 클럽"이 형성되면서 확산되지만, 엉뚱한 인물이 개입하면서 통제할 수 없는 혼돈으로 이어지게 됩니다.
  
- **개봉일**: 1999년 10월 15일  
- **장르**: 드라마, 스릴러  
- **상영 시간**: 139분  
- **예산**: 63,000,000 달러  
- **수익**: 100,853,753 달러  
- **평점**: 8.4/10 (투표 수: 31,732)  
- **태그라인**: "Mischief. Mayhem. Soap."  
- **홈페이지**: [Fight Club](https://www.20thcenturystudios.com/movies/fight-club)

![Fight Club 포스터](https://image.tmdb.org/t/p/w780/jSziioSwPVrOy9Yow3XhWIBDjq1.jpg)

이 영화는 남성 정체성, 소비주의, 정신 건강 등에 대해 깊은 질문을 던지는 작품으로, 시간이 지나도 많은 사랑을 받고 있습니다.


'영화 ID 550에 해당하는 영화는 **"Fight Club"**입니다.  \n\n- **개요**: 한 시간에 대해 고민하는 불면증 환자와 미끄러지듯한 비누 판매자가 남성의 본능적인 공격성을 새로운 형태의 치료법으로 channeling합니다. 그들의 개념은 각 도시마다 underground "격투 클럽"이 형성되면서 확산되지만, 엉뚱한 인물이 개입하면서 통제할 수 없는 혼돈으로 이어지게 됩니다.\n  \n- **개봉일**: 1999년 10월 15일  \n- **장르**: 드라마, 스릴러  \n- **상영 시간**: 139분  \n- **예산**: 63,000,000 달러  \n- **수익**: 100,853,753 달러  \n- **평점**: 8.4/10 (투표 수: 31,732)  \n- **태그라인**: "Mischief. Mayhem. Soap."  \n- **홈페이지**: [Fight Club](https://www.20thcenturystudios.com/movies/fight-club)\n\n![Fight Club 포스터](https://image.tmdb.org/t/p/w780/jSziioSwPVrOy9Yow3XhWIBDjq1.jpg)\n\n이 영화는 남성 정체성, 소비주의, 정신 건강 등에 대해 깊은 질문을 던지는 작품으로, 시간이 지나도 많은 사랑을 받고 있습니다.'

In [19]:
# 테스트 3: 특정 영화 출연진 조회
run_agent('movie ID 550에 해당하는 영화에 누가 출연하는지 알려줘')

사용자: movie ID 550에 해당하는 영화에 누가 출연하는지 알려줘
[함수 호출] get_movie_credits {'id': 550}

에이전트: 영화 ID 550에 해당하는 영화 "파이트 클럽 (Fight Club)"의 주요 출연진은 다음과 같습니다:

1. **에드워드 노턴 (Edward Norton)** - 내레이터
   ![Edward Norton](https://image.tmdb.org/t/p/w185/8nytsqL59SFJTVYVrN72k6qkGgJ.jpg)

2. **브래드 피트 (Brad Pitt)** - 타일러 더든
   ![Brad Pitt](https://image.tmdb.org/t/p/w185/r9DzKQLNbh5QfXlrFGHoVNKER7X.jpg)

3. **헬레나 본햄 카터 (Helena Bonham Carter)** - 말라 싱어
   ![Helena Bonham Carter](https://image.tmdb.org/t/p/w185/hJMbNSPJ2PCahsP3rNEU39C8GWU.jpg)

4. **미트 로프 (Meat Loaf)** - 로버트 폴슨
   ![Meat Loaf](https://image.tmdb.org/t/p/w185/7gKLR1u46OB8WJ6m06LemNBCMx6.jpg)

5. **자레드 레토 (Jared Leto)** - 앤젤 페이스
   ![Jared Leto](https://image.tmdb.org/t/p/w185/ca3x0OfIKbJppZh8S1Alx3GfUZO.jpg)

이 외에도 많은 배우들이 출연하였습니다. 궁금한 점이 있으시면 언제든지 물어보세요!


'영화 ID 550에 해당하는 영화 "파이트 클럽 (Fight Club)"의 주요 출연진은 다음과 같습니다:\n\n1. **에드워드 노턴 (Edward Norton)** - 내레이터\n   ![Edward Norton](https://image.tmdb.org/t/p/w185/8nytsqL59SFJTVYVrN72k6qkGgJ.jpg)\n\n2. **브래드 피트 (Brad Pitt)** - 타일러 더든\n   ![Brad Pitt](https://image.tmdb.org/t/p/w185/r9DzKQLNbh5QfXlrFGHoVNKER7X.jpg)\n\n3. **헬레나 본햄 카터 (Helena Bonham Carter)** - 말라 싱어\n   ![Helena Bonham Carter](https://image.tmdb.org/t/p/w185/hJMbNSPJ2PCahsP3rNEU39C8GWU.jpg)\n\n4. **미트 로프 (Meat Loaf)** - 로버트 폴슨\n   ![Meat Loaf](https://image.tmdb.org/t/p/w185/7gKLR1u46OB8WJ6m06LemNBCMx6.jpg)\n\n5. **자레드 레토 (Jared Leto)** - 앤젤 페이스\n   ![Jared Leto](https://image.tmdb.org/t/p/w185/ca3x0OfIKbJppZh8S1Alx3GfUZO.jpg)\n\n이 외에도 많은 배우들이 출연하였습니다. 궁금한 점이 있으시면 언제든지 물어보세요!'